# DiffusionPen

Genereer synthetische handgeschreven tekst met DiffusionPen (few-shot latent diffusion).  
**Use case:** datasets genereren voor OCR-modellen in de postale sector.

**Structuur:**
1. Setup (dependencies, repo, modellen)
2. Functies
3. Genereer handgeschreven tekst

## 1. Setup
Installeert dependencies, cloned de repo, downloadt modellen, en laadt alles in.

In [1]:
## --- Dependencies ---
#!pip install -q torch torchvision diffusers transformers timm einops omegaconf opencv-python-headless huggingface_hub pillow

## --- Clone repo & download modellen ---
import os, sys, subprocess, shutil, copy, json, random, tempfile
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.nn import DataParallel
from diffusers import AutoencoderKL, DDIMScheduler
from transformers import CanineTokenizer, CanineModel
from PIL import Image, ImageOps
from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt

# Clone DiffusionPen repo — check op een specifiek bronbestand, niet op de map,
# want snapshot_download maakt DiffusionPen/ al aan voor de gewichten.
if not os.path.exists("DiffusionPen/unet.py"):
    os.makedirs("DiffusionPen", exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        clone_dst = os.path.join(tmp, "DiffusionPen")
        subprocess.run(
            ["git", "clone", "https://github.com/koninik/DiffusionPen.git", clone_dst],
            check=True,
        )
        # Kopieer broncode naar DiffusionPen/ zonder bestaande weight-folders te overschrijven
        for name in os.listdir(clone_dst):
            src = os.path.join(clone_dst, name)
            dst = os.path.join("DiffusionPen", name)
            if os.path.exists(dst):
                continue
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
    print("Repo broncode gekopieerd.")
else:
    print("Repo broncode aanwezig.")

# Download model weights van HuggingFace
snapshot_download(
    repo_id="konnik/DiffusionPen",
    local_dir="DiffusionPen/hf_models",
    allow_patterns=["style_models/*", "diffusionpen_iam_model_path/**"],
)
for src, dst in [
    ("DiffusionPen/hf_models/style_models", "DiffusionPen/style_models"),
    ("DiffusionPen/hf_models/diffusionpen_iam_model_path", "DiffusionPen/diffusionpen_iam_model_path"),
]:
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
print("Modellen gedownload.")

# Stel pad in — REPO_DIR moet vóór site-packages staan, anders pakt Python
# een eventueel pip-geïnstalleerd `unet` pakket i.p.v. DiffusionPen's eigen unet.py
REPO_DIR = os.path.abspath("DiffusionPen")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Forceer verse import mocht er nog een pip-pakket in sys.modules hangen van een
# eerdere (mislukte) run — anders geeft Python de gecachete versie terug.
for _name in ("unet", "feature_extractor"):
    sys.modules.pop(_name, None)

# Pas NA sys.path.insert importeren — deze modules wonen in de repo
from unet import UNetModel
from feature_extractor import ImageEncoder

## --- Configuratie ---
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
IMG_SIZE = (64, 256)

class Args:
    device = DEVICE
    interpolation = False
    mix_rate = None
    img_feat = True
    latent = True
    color = True
args = Args()

## --- Laad modellen ---
device_idx = int(DEVICE.split(":")[-1]) if "cuda" in DEVICE else 0
device_ids = [device_idx] if "cuda" in DEVICE else None

tokenizer = CanineTokenizer.from_pretrained("google/canine-c")
text_encoder = CanineModel.from_pretrained("google/canine-c").to(DEVICE)
text_encoder.eval()

vae = DataParallel(AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="vae").to(DEVICE), device_ids=device_ids)
vae.requires_grad_(False); vae.eval()

ddim = DDIMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler")

style_encoder = ImageEncoder(model_name='mobilenetv2_100', num_classes=0, pretrained=True, trainable=True)
sd = torch.load("./style_models/iam_style_diffusionpen.pth", map_location=DEVICE)
md = style_encoder.state_dict()
md.update({k: v for k, v in sd.items() if k in md and md[k].shape == v.shape})
style_encoder.load_state_dict(md)
style_encoder = DataParallel(style_encoder, device_ids=device_ids).to(DEVICE)
style_encoder.requires_grad_(False); style_encoder.eval()

character_classes = list('_!"#&\'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz ')
unet = UNetModel(
    image_size=IMG_SIZE, in_channels=4, model_channels=320, out_channels=4,
    num_res_blocks=1, attention_resolutions=(1, 1), channel_mult=(1, 1),
    num_heads=4, num_classes=339, context_dim=320, vocab_size=len(character_classes),
    text_encoder=DataParallel(text_encoder, device_ids=device_ids), args=args,
)
unet = DataParallel(unet, device_ids=device_ids).to(DEVICE)
unet.load_state_dict(torch.load("./diffusionpen_iam_model_path/models/ckpt.pt", map_location=DEVICE))
unet.eval()

ema_model = copy.deepcopy(unet).eval().requires_grad_(False)
ema_model.load_state_dict(torch.load("./diffusionpen_iam_model_path/models/ema_ckpt.pt", map_location=DEVICE))
ema_model.eval()

c:\Users\karol\OneDrive\Bureaublad\HHS\Jaar 3 Semester 6\Datalab\Prime_Vision\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo broncode aanwezig.


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 504.00it/s]


Modellen gedownload.


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


DataParallel(
  (module): UNetModel(
    (text_encoder): DataParallel(
      (module): CanineModel(
        (char_embeddings): CanineEmbeddings(
          (HashBucketCodepointEmbedder_0): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_1): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_2): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_3): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_4): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_5): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_6): Embedding(16384, 96)
          (HashBucketCodepointEmbedder_7): Embedding(16384, 96)
          (char_position_embeddings): Embedding(16384, 768)
          (token_type_embeddings): Embedding(16, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (initial_char_encoder): CanineEncoder(
          (layer): ModuleList(
            (

## 2. Functies

In [2]:
def crop_whitespace(img_array, padding=2):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY) if len(img_array.shape) == 3 else img_array
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img_array
    x, y, w, h = cv2.boundingRect(coords)
    x, y = max(0, x - padding), max(0, y - padding)
    w = min(img_array.shape[1] - x, w + 2 * padding)
    h = min(img_array.shape[0] - y, h + 2 * padding)
    return img_array[y:y+h, x:x+w]


def ink_mask(gray_img):
    _, binary = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    dilated = cv2.dilate(binary, np.ones((2, 2), np.uint8), iterations=1)
    return cv2.GaussianBlur(dilated.astype(np.float32), (3, 3), 0.8) / 255.0


def ink_coverage(gray_img, threshold=200):
    return np.mean(gray_img < threshold)


def alpha_composite(canvas, patch, mask, x, y):
    ch, cw = canvas.shape
    ph, pw = patch.shape
    # Bereken overlappend gebied (clip aan canvasranden)
    y1, y2 = max(y, 0), min(y + ph, ch)
    x1, x2 = max(x, 0), min(x + pw, cw)
    py1, py2 = y1 - y, y2 - y
    px1, px2 = x1 - x, x2 - x
    if y2 <= y1 or x2 <= x1:
        return
    a = mask[py1:py2, px1:px2]
    canvas[y1:y2, x1:x2] = (1.0 - a) * canvas[y1:y2, x1:x2] + a * patch[py1:py2, px1:px2]


@torch.no_grad()
def generate_word(word, style_id=None, style_images=None, num_steps=50, max_retries=4):
    # Korte woorden (cijfers, "AB") zijn lastiger → meer stappen helpen
    effective_steps = num_steps + 20 if len(word) <= 2 else num_steps

    # Minimale inktdrempel: korte woorden mogen minder inkt hebben, maar niet bijna nul
    min_ink = 0.02 if len(word) <= 2 else 0.04

    # Style features voorbereiden
    if style_images is not None and len(style_images) > 0:
        tf = transforms.Compose([transforms.ToTensor(),
                                  transforms.Normalize((0.5,)*3, (0.5,)*3)])
        st = torch.stack([tf(im.convert("RGB").resize((256, 64))) for im in style_images]).to(DEVICE)
        style_feats = style_encoder(st.reshape(-1, 3, 64, 256)).to(DEVICE)
        labels = torch.tensor([0]).long().to(DEVICE)
        style_tens = st
    else:
        style_tens, style_feats = None, None
        labels = torch.tensor([style_id or 0]).long().to(DEVICE)

    # Tokenize het woord voor text-conditioning
    text_features = tokenizer(word, padding="max_length", truncation=True,
                              return_tensors="pt", max_length=40).to(DEVICE)

    best_result = None
    best_ink = 0.0

    for attempt in range(max_retries):
        # Start van pure ruis
        x = torch.randn((1, 4, IMG_SIZE[0] // 8, IMG_SIZE[1] // 8)).to(DEVICE)

        # DDIM denoising loop
        ddim.set_timesteps(effective_steps)
        for time in ddim.timesteps:
            t = (torch.ones(1) * time.item()).long().to(DEVICE)
            pred = ema_model(x, t, text_features, labels,
                             original_images=style_tens, mix_rate=None,
                             style_extractor=style_feats)
            x = ddim.step(pred, time, x).prev_sample

        # Decodeer latent → pixel via VAE
        latents = 1 / 0.18215 * x
        image = vae.module.decode(latents).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        rgb = (image.cpu().permute(0, 2, 3, 1).numpy()[0] * 255).astype(np.uint8)
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        cropped = crop_whitespace(gray, padding=3)

        # Check of er genoeg inkt is (= of het woord echt gerenderd is)
        coverage = ink_coverage(cropped)

        # Bewaar het beste resultaat
        if coverage > best_ink:
            best_ink = coverage
            best_result = (cropped, ink_mask(cropped))

        # Genoeg inkt → accepteer meteen
        if coverage >= min_ink:
            return cropped, ink_mask(cropped)

    # Geen enkele poging haalde de drempel → geef het beste terug
    return best_result


def generate_line(text, style_id=None, style_images=None, target_h=52):
    words = text.strip().split()
    if not words:
        return np.ones((target_h, 10), dtype=np.float32) * 255.0

    # Genereer elk woord apart en resize naar uniforme hoogte
    word_items = []
    for word in words:
        ink_raw, mask_raw = generate_word(word, style_id=style_id, style_images=style_images)
        h, w = ink_raw.shape
        ar = w / max(h, 1)
        # Breedte bepalen: begrensd op basis van aspect ratio en aantal tekens
        new_w = max(12, int(ar * target_h))
        new_w = np.clip(new_w, int(len(word) * target_h * 0.3), int(len(word) * target_h * 0.9))
        ink_r = cv2.resize(ink_raw, (new_w, target_h), interpolation=cv2.INTER_AREA)
        mask_r = cv2.resize(mask_raw, (new_w, target_h), interpolation=cv2.INTER_LINEAR)
        word_items.append((ink_r, mask_r))

    # Canvas klaarmaken (brede marge voor veiligheid)
    avg_gap = int(target_h * 0.28)
    total_w = sum(ink.shape[1] for ink, _ in word_items) + avg_gap * len(word_items) + 20
    canvas = np.ones((target_h, total_w), dtype=np.float32) * 255.0

    # Woorden plaatsen met variabele tussenruimte en baseline-drift
    cursor = 8
    for i, (ink, mask) in enumerate(word_items):
        if i > 0:
            cursor += random.randint(int(avg_gap * 0.55), int(avg_gap * 1.45))
        v_shift = random.randint(-2, 2)  # lichte verticale verschuiving
        alpha_composite(canvas, ink.astype(np.float32), mask, cursor, v_shift)
        cursor += ink.shape[1]

    return canvas[:, :cursor + 8]


def generate_text(regels, style_id=None, style_images=None,
                  target_h=52, line_gap=6, margin=15,
                  paper_tone=245, ink_darken=0.82, noise_strength=2.5):

    # Genereer alle regels als losse afbeeldingen
    line_imgs = []
    for regel in regels:
        line_imgs.append(generate_line(regel, style_id=style_id,
                                        style_images=style_images, target_h=target_h))

    # Canvas aanmaken met papierkleur
    max_w = max(img.shape[1] for img in line_imgs) + 2 * margin
    total_h = len(line_imgs) * target_h + (len(line_imgs) - 1) * line_gap + 2 * margin
    canvas = np.ones((total_h, max_w), dtype=np.float32) * paper_tone

    # Subtiele papiertextuur via Gaussian noise
    noise = cv2.GaussianBlur(
        np.random.normal(0, noise_strength, (total_h, max_w)).astype(np.float32),
        (7, 7), 2.0
    )
    canvas = np.clip(canvas + noise, 0, 255)

    # elke regel compositen op het canvas
    y = margin
    for line_img in line_imgs:
        h, w = line_img.shape
        x_off = margin + random.randint(-3, 3)  # lichte horizontale variatie

        # Micro-rotatie voor natuurlijk effect
        angle = random.uniform(-0.4, 0.4)
        if abs(angle) > 0.05:
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            line_img = cv2.warpAffine(line_img, M, (w, h),
                                       borderMode=cv2.BORDER_CONSTANT, borderValue=255)

        # Inkt-masker berekenen en inkt donkerder maken
        line_mask = (255.0 - line_img) / 255.0
        darkened = line_img * ink_darken

        # Alpha-blend op canvas
        alpha_composite(canvas, darkened, line_mask, x_off, y)
        y += h + line_gap

    return Image.fromarray(np.clip(canvas, 0, 255).astype(np.uint8))


print("Functies geladen.")


Functies geladen.


## 3. Pipeline: PaddleOCR (detectie) + TrOCR (herkenning)

Op de volle Joep-pagina's (`IMG_7395.jpeg` … `IMG_7399.jpeg`):
1. PaddleOCR detecteert tekstregio's (alleen detectie, geen recognition).
2. Elke crop wordt door TrOCR-handwritten gehaald voor herkenning.

In [ ]:
## Combineer PaddleOCR-detectie met TrOCR-recognition
# Vereist downgrade: pip install "paddlepaddle==2.6.2" "paddleocr==2.7.3"
if "ocr" not in globals():
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
    print("PaddleOCR (v2.x API) geladen.")


# TrOCR initialiseren als dat nog niet is gebeurd
if "trocr_recognize" not in globals():
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(DEVICE).eval()

    @torch.no_grad()
    def trocr_recognize(pil_img):
        pixel_values = trocr_processor(images=pil_img.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)
        ids = trocr_model.generate(pixel_values, max_length=64)
        return trocr_processor.batch_decode(ids, skip_special_tokens=True)[0]
    print("TrOCR geladen.")

# Pad naar Styles/Joep
def _find_joep_dir():
    cur = os.path.abspath(os.getcwd())
    for _ in range(6):
        cand = os.path.join(cur, "Styles", "Joep")
        if os.path.isdir(cand):
            return cand
        parent = os.path.dirname(cur)
        if parent == cur: break
        cur = parent
    return None

JOEP_DIR = _find_joep_dir()
assert JOEP_DIR, "Styles/Joep niet gevonden"

PAGE_FILES = ["IMG_7395.jpeg", "IMG_7396.jpeg", "IMG_7397.jpeg", "IMG_7398.jpeg", "IMG_7399.jpeg"]


def crop_polygon(img_np, box, pad=4):
    """Axis-aligned crop rond een polygoon, met kleine padding."""
    pts = np.array(box, dtype=np.int32)
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)
    h, w = img_np.shape[:2]
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    return img_np[y1:y2, x1:x2]


for fname in PAGE_FILES:
    fpath = os.path.join(JOEP_DIR, fname)
    img_np = np.array(Image.open(fpath).convert("RGB"))

    # Stap 1: detectie via PaddleOCR v2.x (rec=False → alleen bounding boxes)
    det_out = ocr.ocr(img_np, det=True, rec=False, cls=False)
    boxes = det_out[0] if det_out and det_out[0] else []

    # Sorteer top-to-bottom op gemiddelde y van het polygoon
    boxes = sorted(boxes, key=lambda b: float(np.mean([p[1] for p in b])))

    # Stap 2: crop + TrOCR per regio
    preds = []
    for box in boxes:
        crop = crop_polygon(img_np, box)
        if crop.size == 0:
            preds.append("")
            continue
        pred = trocr_recognize(Image.fromarray(crop))
        preds.append(pred)

    # Visualiseer pagina + boxes met index
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img_np)
    for i, box in enumerate(boxes):
        pts = np.array(box)
        ax.plot(np.append(pts[:, 0], pts[0, 0]),
                np.append(pts[:, 1], pts[0, 1]),
                'r-', linewidth=1.5)
        ax.text(pts[0, 0], pts[0, 1] - 6, str(i),
                color='red', fontsize=11, weight='bold')
    ax.set_title(f"{fname} — {len(boxes)} regio's gedetecteerd")
    ax.axis("off")
    plt.tight_layout(); plt.show()

    # Print herkenningen
    print(f"\n=== {fname} ===")
    for i, p in enumerate(preds):
        print(f"  [{i:2d}] {p!r}")

ModuleNotFoundError: No module named 'paddleocr'

: 